## RGI Project Part I - 2025/2026

- Gustavo Brazil (ist1116490)
- João Pires (ist1116478)
- Miguel Soares (ist190494)

---

### 1.1 Preprocessing and Data Loading

Make sure the spaCy English model is installed.

**Text processing pipeline:** Sentence segmentation with pysbd → lemmatisation, stopword/punctuation removal, noun phrase extraction with spacy. Lemmatisation was chosen over stemming to preserve readability and semantic coherence in vector representations.

In [9]:
import os
import spacy
import joblib
from spacy.language import Language
from pysbd.utils import PySBDFactory

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_PATH  = "BBC News Summary"
PROC_FILE  = "processed_data.joblib"

# Load once globally to avoid redundant model loads
nlp = spacy.load("en_core_web_sm")

try:
    @Language.factory("pysbd")
    def create_pysbd_component(nlp, name):
        return PySBDFactory(nlp)
except ValueError:
    pass

nlp.add_pipe("pysbd", first=True)

# ── Preprocessing ─────────────────────────────────────────────────────────────
def preprocess_text(text):
    """Sentence segmentation, lemmatisation, stopword removal, noun-phrase extraction."""

    doc = nlp(text)
    sentences, processed_sentences = [], []

    for sent in doc.sents:
        sentences.append(sent.text.strip())

        tokens = [
            token.lemma_.lower()
            for token in sent
            if not token.is_stop and not token.is_punct and token.text.strip()
        ]
        noun_phrases = [chunk.text.lower() for chunk in sent.noun_chunks]
        processed_sentences.append(tokens + noun_phrases)

    return sentences, processed_sentences

# ── Dataset loader ─────────────────────────────────────────────────────────────
def load_bbc_dataset(base_path):
    """Reads the BBC News folder structure and returns a list of document dicts."""
    categories  = ["business", "entertainment", "politics", "sport", "tech"]
    articles_dir = os.path.join(base_path, "News Articles")
    summaries_dir = os.path.join(base_path, "Summaries")
    data = []

    for cat in categories:
        cat_path = os.path.join(articles_dir, cat)
        if not os.path.exists(cat_path):
            continue
        for filename in sorted(f for f in os.listdir(cat_path) if f.endswith(".txt")):
            art_path = os.path.join(cat_path, filename)
            sum_path = os.path.join(summaries_dir, cat, filename)
            with open(art_path, "r", encoding="latin1") as fa,                  open(sum_path,  "r", encoding="latin1") as fs:
                content     = fa.read()
                ref_summary = fs.read()

            sents, proc_sents = preprocess_text(content)
            data.append({
                "doc_id":              f"{cat}_{filename.split('.')[0]}",
                "category":            cat,
                "original_text":       content,
                "sentences":           sents,
                "processed_sentences": proc_sents,
                "reference_summary":   ref_summary,
            })
    return data

# ── Main ───────────────────────────────────────────────────────────────────────
if os.path.exists(PROC_FILE):
    print(f"Loading pre-processed data from {PROC_FILE}...")
    D = joblib.load(PROC_FILE)
else:
    print("Starting dataset loading and NLP preprocessing...")
    D = load_bbc_dataset(BASE_PATH)
    joblib.dump(D, PROC_FILE)
    print("Processing complete.")

print(f"Successfully loaded {len(D)} documents.")

Loading pre-processed data from processed_data.joblib...
Successfully loaded 2225 documents.


### 1.2 Indexing Function

The hybrid index contains:
- **Inverted index** (lexical / sparse) — stores per-document term frequencies  
- **Embeddings** (dense / semantic) — one vector per sentence, built with **two** encoder LLMs with distinct profiles:
  - `all-MiniLM-L6-v2` — general-purpose bi-encoder, trained with contrastive learning on diverse corpora  
  - `multi-qa-MiniLM-L6-cos-v1` — fine-tuned specifically for semantic similarity and question-answering tasks

Using two encoders lets us compare sparse vs. dense representations and assess whether domain-specific fine-tuning affects summarisation quality.

* doc_lengths is useless
* change the way the text embeddings are made

In [ ]:
import time, sys, math, os, joblib
import numpy as np
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()

# ── Load HuggingFace token from environment (never hardcode credentials) ───────
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(HF_TOKEN)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# ── Two encoder LLMs with distinct profiles ───────────────────────────────────
ENCODER_MODELS = {
    "minilm_general": "all-MiniLM-L6-v2",          # general-purpose contrastive encoder
    "minilm_qa":      "multi-qa-MiniLM-L6-cos-v1", # QA/similarity fine-tuned encoder
}

def indexing(D, args=None, filename="hybrid_index.joblib"):
    """
    Build a hybrid index (inverted + embeddings for 2 encoders).

    Returns
    -------
    I            : dict  — the index
    indexing_time: float — wall-clock seconds (0 if loaded from cache)
    size_bytes   : int   — sys.getsizeof(I)
    """
    if os.path.exists(filename):
        print(f"Loading existing index from {filename}...")
        I = joblib.load(filename)
        return I, 0.0, sys.getsizeof(I)

    print("Index not found. Building index...")
    start = time.time()

    # ── Lexical (inverted) index ───────────────────────────────────────────────
    inverted_index = defaultdict(lambda: defaultdict(int))
    doc_lengths    = {}

    # Pre-load both encoders once
    encoders = {key: SentenceTransformer(name) for key, name in ENCODER_MODELS.items()}
    sentence_embeddings = {key: {} for key in ENCODER_MODELS}

    for doc in D:
        doc_id           = doc["doc_id"]
        all_terms_in_doc = []

        # Dense: encode sentences with both models

        # this thing is creating embeddings for each and every sentence of the texts
        # for embeddings of short texts it is better to feed the whole natural language text

        for enc_key, model in encoders.items():
            sentence_embeddings[enc_key][doc_id] = model.encode(doc["sentences"])

        # Sparse: populate inverted index
        for sent_terms in doc["processed_sentences"]:
            for term in sent_terms:
                inverted_index[term][doc_id] += 1
                all_terms_in_doc.append(term)

        doc_lengths[doc_id] = len(all_terms_in_doc)

    indexing_time = time.time() - start

    I = {
        "inverted_index":    dict(inverted_index),
        "doc_lengths":       doc_lengths,
        "embeddings":        sentence_embeddings,   # {"minilm_general": {doc_id: array}, ...}
        "total_docs":        len(D),
        "encoder_models":    ENCODER_MODELS,
        # Collection-level avg sentence length (needed for BM25)
        "avg_sent_len":      np.mean([
            len(s) for doc in D for s in doc["processed_sentences"]
        ]),
    }

    joblib.dump(I, filename)
    print(f"Index built in {indexing_time:.1f}s and saved to {filename}.")
    return I, indexing_time, sys.getsizeof(I)

I, itime, isize = indexing(D)
print(f"Indexing time: {itime:.1f}s | In-memory size: {isize/1e6:.1f} MB")

Index not found. Building index...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2294.96it/s]


Index built in 1761.2s and saved to hybrid_index.joblib.
Indexing time: 1761.2s | In-memory size: 0.0 MB


In [36]:
len(D[0]["sentences"])

21

In [33]:
I["embeddings"]["minilm_general"]["business_001"]

array([[-0.03835344,  0.00612135, -0.0113062 , ..., -0.0680441 ,
         0.00720908, -0.03758336],
       [ 0.0389058 , -0.01313104, -0.01203297, ..., -0.15502392,
        -0.04560921,  0.02405163],
       [-0.02079609, -0.09069284,  0.0242595 , ..., -0.02909165,
         0.03828218,  0.02349396],
       ...,
       [ 0.02793075,  0.08059135, -0.03631466, ..., -0.07346802,
        -0.0057332 , -0.01138691],
       [-0.02377516, -0.02278955, -0.04182864, ..., -0.07620808,
         0.00329625, -0.00306832],
       [ 0.01119744,  0.0110287 , -0.01522651, ..., -0.14092335,
        -0.05764708,  0.02414384]], shape=(21, 384), dtype=float32)

In [25]:
#inverted index
i_i = dict()

for doc in D:
    doc_id = doc["doc_id"]

    # Sparse: populate inverted index
    for sent_terms in doc["processed_sentences"]:
        for term in sent_terms:

            i_i[term] = i_i.get(term,dict())
            i_i[term][doc_id] = i_i[term].get(doc_id,0) + 1

i_i


{'ad': {'business_001': 1,
  'business_011': 1,
  'business_035': 1,
  'business_443': 1,
  'entertainment_041': 1,
  'entertainment_173': 1,
  'entertainment_183': 1,
  'politics_303': 1,
  'politics_332': 2,
  'politics_346': 2,
  'politics_411': 1,
  'tech_081': 1,
  'tech_089': 1,
  'tech_096': 2,
  'tech_124': 2,
  'tech_138': 1,
  'tech_149': 1,
  'tech_162': 1,
  'tech_177': 1,
  'tech_206': 4,
  'tech_209': 1,
  'tech_250': 1,
  'tech_267': 1,
  'tech_292': 1,
  'tech_310': 1,
  'tech_364': 1,
  'tech_377': 2,
  'tech_393': 1},
 'sale': {'business_001': 5,
  'business_003': 4,
  'business_004': 1,
  'business_011': 2,
  'business_013': 6,
  'business_014': 1,
  'business_016': 1,
  'business_025': 1,
  'business_026': 1,
  'business_027': 1,
  'business_038': 3,
  'business_039': 2,
  'business_040': 5,
  'business_049': 1,
  'business_050': 6,
  'business_051': 2,
  'business_053': 1,
  'business_054': 2,
  'business_056': 9,
  'business_063': 15,
  'business_067': 1,
  'busin

In [26]:
i_i["happy"]

{'business_020': 1,
 'business_097': 1,
 'business_131': 1,
 'business_162': 1,
 'business_188': 1,
 'business_236': 1,
 'business_252': 1,
 'business_257': 1,
 'business_344': 1,
 'business_363': 1,
 'business_378': 1,
 'business_440': 1,
 'business_452': 1,
 'entertainment_025': 1,
 'entertainment_094': 1,
 'entertainment_180': 2,
 'entertainment_181': 1,
 'entertainment_187': 2,
 'entertainment_203': 1,
 'entertainment_213': 1,
 'entertainment_221': 1,
 'entertainment_225': 1,
 'entertainment_228': 1,
 'entertainment_245': 1,
 'entertainment_253': 3,
 'entertainment_255': 1,
 'entertainment_301': 1,
 'entertainment_337': 1,
 'entertainment_358': 1,
 'entertainment_368': 2,
 'politics_053': 1,
 'politics_060': 1,
 'politics_088': 1,
 'politics_104': 2,
 'politics_113': 1,
 'politics_126': 1,
 'politics_250': 1,
 'politics_260': 1,
 'politics_262': 1,
 'politics_264': 1,
 'politics_272': 2,
 'politics_274': 1,
 'politics_303': 1,
 'politics_337': 1,
 'politics_380': 1,
 'politics_396'

---

### 2.1 Keyword Extraction Function 

Extracts the top-`p` informative keywords using **TF-IDF** scoring. A semantic redundancy filter (cosine similarity) removes near-duplicate terms. The output preserves the original TF-IDF ranking order as required by the spec (`ordered set of p keywords`).

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

def keyword_extraction(d, p, I, args=None):
    """
    Extract the top-p informative keywords from document d.

    Returns
    -------
    List of p keyword strings ordered by descending TF-IDF score.
    """
    method = (args or {}).get("method", "tfidf")
    N      = I["total_docs"]

    if method == "tfidf":
        # ── Score every term in the document ──────────────────────────────────
        all_terms   = [t for sent in d["processed_sentences"] for t in sent]
        term_counts = Counter(all_terms)

        scores = {}
        for term, tf in term_counts.items():
            df = len(I["inverted_index"].get(term, {}))
            if df == 0:
                continue
            idf            = np.log(N / (df + 1))
            scores[term]   = tf * idf

        # Candidates ordered by score descending — order is preserved throughout
        candidates = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        # ── Redundancy filter (semantic similarity) ────────────────────────────
        enc_key   = (args or {}).get("enc_key", "minilm_general")
        enc_name  = I["encoder_models"].get(enc_key, "all-MiniLM-L6-v2")
        model     = SentenceTransformer(enc_name)
        threshold = (args or {}).get("similarity_threshold", 0.7)

        selected_kws, selected_vecs = [], []
        for term, score in candidates:
            if len(selected_kws) >= p:
                break
            vec = model.encode([term])
            if not selected_kws:
                selected_kws.append(term)
                selected_vecs.append(vec)
            else:
                sims = cosine_similarity(vec, np.vstack(selected_vecs))
                if np.max(sims) < threshold:
                    selected_kws.append(term)
                    selected_vecs.append(vec)

        # Return in original TF-IDF rank order (already maintained above)
        return selected_kws

    elif method == "prompting":
        llm_pipeline = (args or {}).get("llm_pipeline")
        if not llm_pipeline:
            return ["Error: no LLM pipeline provided"]

        text_input = d["original_text"][:600]
        prompt     = (
            f"Extract exactly {p} keywords from the article below.\n"
            f"Return only a comma-separated list, no explanations.\n\n"
            f"Article: {text_input}\n\nKeywords:"
        )
        result = llm_pipeline(
            prompt, max_new_tokens=30, do_sample=False,
            pad_token_id=llm_pipeline.tokenizer.eos_token_id
        )
        generated = result[0]["generated_text"].replace(prompt, "").strip()
        keywords  = [k.strip().lower() for k in generated.replace("\n", ",").split(",")]
        return [k for k in keywords if k][:p]

    return []

### 2.2 Decoder LLMs for Keyword Extraction (Generative approach)

We use **two** lightweight autoregressive decoder LLMs with distinct profiles:

| Model | Architecture | Pretraining focus |
|---|---|---|
| `sshleifer/distilbart-cnn-6-6` | DistilBART (seq2seq) | CNN/DailyMail news summarisation |
| `facebook/opt-125m` | OPT (causal decoder) | General-purpose language modelling |

The two models have different architectures (seq2seq vs causal) and different pretraining tasks, satisfying the "distinct profiles" requirement.

In [ ]:
from transformers import pipeline
import os, joblib
from tqdm import tqdm

# ── Two decoder LLMs ──────────────────────────────────────────────────────────
DECODER_MODELS = {
    "distilbart": "sshleifer/distilbart-cnn-6-6",  # seq2seq, news-summarisation focused
    "opt":        "facebook/opt-125m",              # causal decoder, general-purpose
}

# Initialise pipelines (device=-1 forces CPU)
decoder_pipelines = {}
for key, model_name in DECODER_MODELS.items():
    print(f"Loading {model_name}...")
    task = "summarization" if key == "distilbart" else "text-generation"
    decoder_pipelines[key] = pipeline(task, model=model_name, device=-1)

print("Both decoder LLMs loaded.")

In [ ]:
LLM_KEYWORDS_FILE = "llm_keywords_results.joblib"

if os.path.exists(LLM_KEYWORDS_FILE):
    all_llm_keywords = joblib.load(LLM_KEYWORDS_FILE)
    print(f"Resuming from {len(all_llm_keywords)} processed documents.")
else:
    all_llm_keywords = {}

# Test on first 5 documents; replace [:5] with D to run on full collection
docs_to_process = [d for d in D if d["doc_id"] not in all_llm_keywords][:5]

if docs_to_process:
    for model_key, llm_pipe in decoder_pipelines.items():
        args_llm = {"method": "prompting", "llm_pipeline": llm_pipe}
        for doc in tqdm(docs_to_process, desc=f"Keywords — {model_key}"):
            doc_id = doc["doc_id"]
            if doc_id not in all_llm_keywords:
                all_llm_keywords[doc_id] = {}
            all_llm_keywords[doc_id][model_key] = keyword_extraction(doc, 5, I, args_llm)

    joblib.dump(all_llm_keywords, LLM_KEYWORDS_FILE)
    print("Done.")
else:
    print("All documents already processed.")

In [ ]:
# ── Inspect LLM keyword outputs ───────────────────────────────────────────────
for i, (doc_id, kw_dict) in enumerate(all_llm_keywords.items()):
    print(f"Document: {doc_id}")
    for model_key, kws in kw_dict.items():
        print(f"  [{model_key}] {kws}")
    print("-" * 50)
    if i >= 2:
        break

### 2.3 Summarization Function

Supports four extraction modes (`tfidf`, `bm25`, `embeddings`, `rrf`) plus `prompting`.

**MMR correction:** the formula from the paper is  
$MMR(s) = (1-\lambda)\cdot sim(s,d) - \lambda \sum_{v\in S} sim(s,v)$ 
where **higher λ → more diversity** (λ=1 = maximum anti-redundancy). This is now implemented correctly.

**RRF correction:** the spec fixes `µ = 5` (previously `k=60`).

In [ ]:
import math
import numpy as np
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity

# ── Sentence scoring ──────────────────────────────────────────────────────────
def get_sentence_scores(d, D_collection, I, method="bm25", k1=1.2, b=0.75, enc_key="minilm_general"):
    """
    Score every sentence in document d.

    Parameters
    ----------
    D_collection : list — full corpus (needed for BM25 avg sentence length)
    enc_key      : str  — which encoder embedding to use ('minilm_general' or 'minilm_qa')
    """
    doc_id         = d["doc_id"]
    sentences_terms = d["processed_sentences"]
    N              = I["total_docs"]
    scores         = []

    if method in ("tfidf", "bm25"):
        # Use pre-computed avg sentence length stored in the index
        avgsl = I["avg_sent_len"]

        for sent_terms in sentences_terms:
            sent_len = len(sent_terms)
            if sent_len == 0:
                scores.append(0.0)
                continue

            s_score = 0.0
            counts  = Counter(sent_terms)
            for term, tf in counts.items():
                df = len(I["inverted_index"].get(term, {}))
                if df == 0:
                    continue
                idf = math.log((N - df + 0.5) / (df + 0.5) + 1.0)

                if method == "tfidf":
                    s_score += idf * tf
                else:  # BM25
                    num      = tf * (k1 + 1)
                    den      = tf + k1 * (1 - b + b * (sent_len / avgsl))
                    s_score += idf * (num / den)
            scores.append(s_score)

    elif method == "embeddings":
        embs    = I["embeddings"][enc_key][doc_id]
        doc_vec = np.mean(embs, axis=0).reshape(1, -1)
        scores  = cosine_similarity(embs, doc_vec).flatten().tolist()

    return np.array(scores)

# ── RRF fusion (µ=5 as per spec) ─────────────────────────────────────────────
def rrf_fusion(rankings, mu=5):
    """Combine multiple sentence score arrays via Reciprocal Rank Fusion."""
    n              = len(rankings[0])
    rrf_scores     = np.zeros(n)
    for r_scores in rankings:
        # rank 0 = best
        ranks = n - 1 - np.argsort(np.argsort(r_scores))
        for i, rank in enumerate(ranks):
            rrf_scores[i] += 1.0 / (mu + rank)
    return rrf_scores

# ── MMR selection (formula matches spec: (1−λ)·sim(s,d) − λ·Σsim(s,v)) ───────
def mmr_selection(doc_id, sentence_scores, I, p, lambd=0.5, enc_key="minilm_general"):
    """
    Select p sentences via Maximal Marginal Relevance.

    λ=0 → purely relevance-driven (no diversity)
    λ=1 → maximum diversity (minimum redundancy)
    """
    embs             = I["embeddings"][enc_key][doc_id]
    selected_indices = []
    remaining        = list(range(len(sentence_scores)))

    max_s = np.max(sentence_scores)
    min_s = np.min(sentence_scores)
    norm_scores = (
        (sentence_scores - min_s) / (max_s - min_s)
        if max_s != min_s
        else np.zeros_like(sentence_scores)
    )

    for _ in range(min(p, len(sentence_scores))):
        mmr_scores = []
        for idx in remaining:
            relevance  = norm_scores[idx]
            if not selected_indices:
                redundancy = 0.0
            else:
                sims       = cosine_similarity(
                    embs[idx].reshape(1, -1), embs[selected_indices]
                )
                redundancy = float(np.sum(sims))   # sum, not max, as per formula

            # Correct formula: (1−λ)·relevance − λ·redundancy
            mmr_scores.append((1 - lambd) * relevance - lambd * redundancy)

        best_in_remaining = int(np.argmax(mmr_scores))
        best_global       = remaining.pop(best_in_remaining)
        selected_indices.append(best_global)

    return selected_indices

# ── Main summarization orchestrator ──────────────────────────────────────────
def summarization(d, p, I, D_collection, args=None, l=None):
    """
    Summarise document d.

    Parameters
    ----------
    p            : int  — max number of sentences (extractive) or tokens (generative)
    l            : int  — max characters (optional, applied post-selection)
    D_collection : list — full corpus (passed through to BM25)
    args         : dict — method, lambda, enc_key, llm_pipeline, …

    Returns
    -------
    Extractive : list of (sentence_position, score) tuples ordered by position
    Generative : list with a single summary string
    """
    args   = args or {}
    method = args.get("method", "rrf")
    lambd  = args.get("lambda", 0.5)
    enc_key = args.get("enc_key", "minilm_general")

    # ── Extractive ────────────────────────────────────────────────────────────
    if method in ("tfidf", "bm25", "embeddings", "rrf"):
        if method == "rrf":
            s_bm25   = get_sentence_scores(d, D_collection, I, method="bm25",       enc_key=enc_key)
            s_emb    = get_sentence_scores(d, D_collection, I, method="embeddings",  enc_key=enc_key)
            scores   = rrf_fusion([s_bm25, s_emb])
        else:
            scores   = get_sentence_scores(d, D_collection, I, method=method, enc_key=enc_key)

        indices  = mmr_selection(d["doc_id"], scores, I, p, lambd=lambd, enc_key=enc_key)
        indices_sorted = sorted(indices)   # chronological order

        # Build output: ordered list of (position, score) pairs
        result = [(i, float(scores[i])) for i in indices_sorted]

        # Apply character limit if requested
        if l is not None:
            filtered, total = [], 0
            for pos, score in result:
                sent = d["sentences"][pos]
                if total + len(sent) <= l:
                    filtered.append((pos, score))
                    total += len(sent)
            result = filtered

        return result   # list of (position, score)

    # ── Generative ────────────────────────────────────────────────────────────
    elif method == "prompting":
        llm_pipeline = args.get("llm_pipeline")
        model_key    = args.get("model_key", "distilbart")

        if not llm_pipeline:
            return [("Error: no LLM pipeline provided", 0.0)]

        text_input = d["original_text"][:800]

        if model_key == "distilbart":
            # seq2seq summarisation model — use summarization pipeline directly
            result = llm_pipeline(
                text_input,
                max_length=130, min_length=30, do_sample=False
            )
            summary = result[0]["summary_text"]
        else:
            # causal decoder — use prompt engineering
            prompt  = (
                f"Summarize the following news article in 2-3 sentences:\n\n"
                f"{text_input}\n\nSummary:"
            )
            result  = llm_pipeline(
                prompt, max_new_tokens=80, do_sample=False,
                pad_token_id=llm_pipeline.tokenizer.eos_token_id
            )
            summary = result[0]["generated_text"].replace(prompt, "").strip()

        return [summary]  # single string in a list

### 2.4 Generate All Summaries

In [ ]:
import os, joblib
from tqdm import tqdm

SUMMARIES_FILE = "summaries_results.joblib"
summary_args   = {"method": "rrf", "lambda": 0.5, "enc_key": "minilm_general"}
P              = 3   # sentences per summary

if os.path.exists(SUMMARIES_FILE):
    print(f"Loading summaries from {SUMMARIES_FILE}...")
    all_summaries = joblib.load(SUMMARIES_FILE)
else:
    print("Generating extractive summaries (RRF + MMR)...")
    all_summaries = {}
    try:
        for doc in tqdm(D, desc="Summarising"):
            all_summaries[doc["doc_id"]] = summarization(doc, P, I, D, summary_args)
        joblib.dump(all_summaries, SUMMARIES_FILE)
        print("Done.")
    except KeyboardInterrupt:
        joblib.dump(all_summaries, SUMMARIES_FILE)
        print("Interrupted. Progress saved.")

# Quick sanity check
doc0 = D[0]
sample = all_summaries[doc0["doc_id"]]
print("\nSample extractive summary (position, score):")
for pos, score in sample:
    print(f"  [{pos}] (score={score:.4f}) {doc0['sentences'][pos][:120]}...")

### 3. Generative Summarisation

We run both decoder LLMs on the first 10 documents as a demonstration. Replace `[:10]` with `D` to run on the full collection.

In [ ]:
GEN_SUMMARIES_FILE = "summaries_generative_results.joblib"

if os.path.exists(GEN_SUMMARIES_FILE):
    all_gen_summaries = joblib.load(GEN_SUMMARIES_FILE)
    print(f"Loaded {len(all_gen_summaries)} generative summaries.")
else:
    all_gen_summaries = {}

docs_to_gen = [d for d in D if d["doc_id"] not in all_gen_summaries][:10]

if docs_to_gen:
    for model_key, llm_pipe in decoder_pipelines.items():
        gen_args = {"method": "prompting", "llm_pipeline": llm_pipe, "model_key": model_key}
        for doc in tqdm(docs_to_gen, desc=f"Generative — {model_key}"):
            doc_id = doc["doc_id"]
            if doc_id not in all_gen_summaries:
                all_gen_summaries[doc_id] = {}
            all_gen_summaries[doc_id][model_key] = summarization(doc, P, I, D, gen_args)
    joblib.dump(all_gen_summaries, GEN_SUMMARIES_FILE)
    print("Done.")
else:
    print("Nothing to process.")

# ── Quick comparison ──────────────────────────────────────────────────────────
ex_id  = list(all_gen_summaries.keys())[0]
ex_doc = next(d for d in D if d["doc_id"] == ex_id)

print(f"\n--- Document: {ex_id} ---")
print("\nEXTRACTIVE (RRF+MMR):")
for pos, score in all_summaries.get(ex_id, []):
    print(f"  {ex_doc['sentences'][pos]}")

for model_key in DECODER_MODELS:
    print(f"\nGENERATIVE ({model_key}):")
    print(f"  {all_gen_summaries[ex_id].get(model_key, ['N/A'])[0]}")

### 4. Evaluation Function

**Extractive summaries** are evaluated against the reference extracts using:
- **Precision-Recall curve** and **AUPRC** (area under uninterpolated PR curve) — over varying summary lengths  
- **Fβ-measure** at a fixed summary length (`p`) — with β=1 (equal precision/recall) unless overridden  

A sentence is considered a *hit* if it appears (exact or normalised match) in the reference extract.

**Generative summaries** are evaluated via **cosine similarity of sentence embeddings** between the generated text and the reference extract, using an independent lightweight encoder (`all-MiniLM-L6-v2`).

In [ ]:
import numpy as np
from sklearn.metrics import auc
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Independent encoder for generative evaluation (not used in indexing)
_eval_encoder = None

def _get_eval_encoder():
    global _eval_encoder
    if _eval_encoder is None:
        _eval_encoder = SentenceTransformer("all-MiniLM-L6-v2")
    return _eval_encoder


def _normalise(text):
    """Lower-case and strip whitespace for sentence matching."""
    return text.strip().lower()


def _sentences_of_reference(ref_text):
    """Split a reference summary into individual sentences."""
    doc = nlp(ref_text)
    return [_normalise(s.text) for s in doc.sents if s.text.strip()]


def _precision_recall_at_k(doc, scored_pairs, ref_sentences, k):
    """
    Compute P@k and R@k for an extractive summary of size k.

    scored_pairs: list of (position, score) — full ranked list (all sentences)
    ref_sentences: set of normalised reference sentences
    """
    # Sort by score desc to pick top-k
    ranked = sorted(scored_pairs, key=lambda x: x[1], reverse=True)[:k]
    selected_sents = {_normalise(doc["sentences"][pos]) for pos, _ in ranked}

    tp = len(selected_sents & ref_sentences)
    p  = tp / k              if k              > 0 else 0.0
    r  = tp / len(ref_sentences) if ref_sentences else 0.0
    return p, r


def evaluation(Sset, Rset, D_collection, I, args=None):
    """
    Evaluate produced summaries against reference extracts.

    Parameters
    ----------
    Sset : dict  {doc_id -> list of (pos, score) for extractive,
                           or list with a single string for generative}
    Rset : dict  {doc_id -> reference_summary_string}
             or list of dicts from D with 'doc_id' and 'reference_summary'
    D_collection : list of document dicts (to recover sentence text)
    I    : hybrid index
    args : dict
        'mode'     : 'extractive' | 'generative' (default 'extractive')
        'p'        : fixed summary length for Fβ   (default 3)
        'beta'     : β for Fβ-measure              (default 1.0)
        'enc_key'  : encoder to use for embedding eval (default 'minilm_general')

    Returns
    -------
    dict with keys: precision, recall, f_beta, auprc  (extractive)
               or   cosine_sim_mean, cosine_sim_std    (generative)
    """
    args     = args or {}
    mode     = args.get("mode", "extractive")
    p_fixed  = args.get("p", 3)
    beta     = args.get("beta", 1.0)

    # Normalise Rset to a dict
    if isinstance(Rset, list):
        Rset = {d["doc_id"]: d["reference_summary"] for d in Rset}

    doc_lookup = {d["doc_id"]: d for d in D_collection}

    results = {}

    # ── EXTRACTIVE ────────────────────────────────────────────────────────────
    if mode == "extractive":
        all_precisions, all_recalls, f_betas, auprcs = [], [], [], []

        for doc_id, scored_pairs in Sset.items():
            if doc_id not in Rset or doc_id not in doc_lookup:
                continue
            doc          = doc_lookup[doc_id]
            ref_sents    = set(_sentences_of_reference(Rset[doc_id]))
            n_sents      = len(doc["sentences"])
            if not ref_sents or n_sents == 0:
                continue

            # ── PR curve over all possible summary lengths ────────────────────
            prec_list, rec_list = [], []
            for k in range(1, n_sents + 1):
                p_k, r_k = _precision_recall_at_k(doc, scored_pairs, ref_sents, k)
                prec_list.append(p_k)
                rec_list.append(r_k)

            # Uninterpolated AUPRC (trapezoid over unique recall levels)
            sorted_pairs = sorted(zip(rec_list, prec_list))
            rec_sorted   = [r for r, _ in sorted_pairs]
            pre_sorted   = [p for _, p in sorted_pairs]
            if len(set(rec_sorted)) > 1:
                auprcs.append(auc(rec_sorted, pre_sorted))

            all_precisions.append(prec_list)
            all_recalls.append(rec_list)

            # ── Fβ at fixed length p ──────────────────────────────────────────
            p_p, r_p = _precision_recall_at_k(doc, scored_pairs, ref_sents, p_fixed)
            denom    = (1 + beta**2) * p_p + beta**2 * r_p  # avoids divide-by-zero
            fb       = (1 + beta**2) * p_p * r_p / denom if denom > 0 else 0.0
            f_betas.append(fb)

        # Aggregate across documents (pad PR curves to same length)
        max_len   = max(len(p) for p in all_precisions) if all_precisions else 0
        avg_prec  = np.mean(
            [np.pad(p, (0, max_len - len(p)), "edge") for p in all_precisions], axis=0
        ) if all_precisions else np.array([])
        avg_rec   = np.mean(
            [np.pad(r, (0, max_len - len(r)), "edge") for r in all_recalls], axis=0
        ) if all_recalls else np.array([])

        results = {
            "avg_precision_curve": avg_prec,
            "avg_recall_curve":    avg_rec,
            "f_beta_mean":         float(np.mean(f_betas)) if f_betas else 0.0,
            "f_beta_std":          float(np.std(f_betas))  if f_betas else 0.0,
            "auprc_mean":          float(np.mean(auprcs))  if auprcs  else 0.0,
            "auprc_std":           float(np.std(auprcs))   if auprcs  else 0.0,
            "n_docs":              len(f_betas),
        }

    # ── GENERATIVE ────────────────────────────────────────────────────────────
    elif mode == "generative":
        encoder = _get_eval_encoder()
        sims    = []

        for doc_id, gen_summary in Sset.items():
            if doc_id not in Rset:
                continue
            # gen_summary is a list; take the first element
            gen_text = gen_summary[0] if isinstance(gen_summary, list) else gen_summary
            ref_text = Rset[doc_id]
            if not gen_text.strip() or not ref_text.strip():
                continue

            gen_vec = encoder.encode([gen_text])
            ref_vec = encoder.encode([ref_text])
            sim     = float(cosine_similarity(gen_vec, ref_vec)[0][0])
            sims.append(sim)

        results = {
            "cosine_sim_mean": float(np.mean(sims)) if sims else 0.0,
            "cosine_sim_std":  float(np.std(sims))  if sims else 0.0,
            "n_docs":          len(sims),
        }

    return results

### 4.1 Run Evaluation — Full Collection & Per-Category

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Rset_all = {d["doc_id"]: d["reference_summary"] for d in D}
eval_args = {"mode": "extractive", "p": 3, "beta": 1.0}

# ── Full-collection evaluation ────────────────────────────────────────────────
print("=== Full collection ===")
full_eval = evaluation(all_summaries, Rset_all, D, I, eval_args)
print(f"  Fβ (β=1, p=3) : {full_eval['f_beta_mean']:.4f} ± {full_eval['f_beta_std']:.4f}")
print(f"  AUPRC          : {full_eval['auprc_mean']:.4f} ± {full_eval['auprc_std']:.4f}")
print(f"  Documents      : {full_eval['n_docs']}")

# ── Per-category evaluation ───────────────────────────────────────────────────
categories = ["business", "entertainment", "politics", "sport", "tech"]
cat_results = {}

print("\n=== Per-category ===")
for cat in categories:
    cat_docs = [d for d in D if d["category"] == cat]
    cat_ids  = {d["doc_id"] for d in cat_docs}
    S_cat    = {k: v for k, v in all_summaries.items() if k in cat_ids}
    R_cat    = {k: v for k, v in Rset_all.items()      if k in cat_ids}

    cat_eval = evaluation(S_cat, R_cat, cat_docs, I, eval_args)
    cat_results[cat] = cat_eval
    print(f"  {cat:<15} Fβ={cat_eval['f_beta_mean']:.4f}±{cat_eval['f_beta_std']:.4f}  "
          f"AUPRC={cat_eval['auprc_mean']:.4f}±{cat_eval['auprc_std']:.4f}  "
          f"(n={cat_eval['n_docs']})")

# ── Plot PR curve for the full collection ─────────────────────────────────────
prec = full_eval["avg_precision_curve"]
rec  = full_eval["avg_recall_curve"]

if len(prec) > 0:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(rec, prec, lw=1.8, label=f"RRF+MMR  (AUPRC={full_eval['auprc_mean']:.3f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve — Full Collection (Extractive)")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("pr_curve_full.png", dpi=120)
    plt.show()
    print("\nPR curve saved to pr_curve_full.png")

### 4.2 Evaluate Generative Summaries (Embedding-based)

In [ ]:
gen_eval_args = {"mode": "generative"}

print("=== Generative evaluation (cosine similarity with reference) ===")
for model_key in DECODER_MODELS:
    # Build Sset for this specific model
    S_gen = {doc_id: sums[model_key]
             for doc_id, sums in all_gen_summaries.items()
             if model_key in sums}
    gen_eval = evaluation(S_gen, Rset_all, D, I, gen_eval_args)
    print(f"  {model_key:<15} cos_sim={gen_eval['cosine_sim_mean']:.4f} "
          f"± {gen_eval['cosine_sim_std']:.4f}  (n={gen_eval['n_docs']})")

### 4.3 MMR — Impact of λ on a Random Document

In [ ]:
import random
random.seed(42)
random_doc = random.choice(D)
print(f"Selected document: {random_doc['doc_id']}")

lambdas = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
lambda_results = {}

for lam in lambdas:
    args_lam = {"method": "bm25", "lambda": lam, "enc_key": "minilm_general"}
    scored   = summarization(random_doc, 3, I, D, args_lam)
    # Evaluate against reference
    S_single = {random_doc["doc_id"]: scored}
    R_single = {random_doc["doc_id"]: random_doc["reference_summary"]}
    res      = evaluation(S_single, R_single, [random_doc], I,
                          {"mode": "extractive", "p": 3, "beta": 1.0})
    lambda_results[lam] = res

print("\nλ    | Fβ    | AUPRC")
print("-" * 30)
for lam, res in lambda_results.items():
    print(f"{lam:.2f} | {res['f_beta_mean']:.4f} | {res['auprc_mean']:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fb_vals    = [lambda_results[lam]["f_beta_mean"] for lam in lambdas]
auprc_vals = [lambda_results[lam]["auprc_mean"]  for lam in lambdas]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(lambdas, fb_vals,    marker="o", label="Fβ (β=1, p=3)")
ax.plot(lambdas, auprc_vals, marker="s", label="AUPRC")
ax.set_xlabel("λ  (higher = more diversity)")
ax.set_ylabel("Score")
ax.set_title(f"MMR λ study — {random_doc['doc_id']}")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("mmr_lambda_study.png", dpi=120)
plt.show()
print("Plot saved to mmr_lambda_study.png")

### 4.4 RRF Impact — BM25 vs Embeddings vs Consensus

In [ ]:
configs = {
    "BM25 only":       {"method": "bm25",       "lambda": 0.5, "enc_key": "minilm_general"},
    "Embeddings only": {"method": "embeddings",  "lambda": 0.5, "enc_key": "minilm_general"},
    "RRF (consensus)": {"method": "rrf",         "lambda": 0.5, "enc_key": "minilm_general"},
    "Embeddings (QA)": {"method": "embeddings",  "lambda": 0.5, "enc_key": "minilm_qa"},
    "RRF (QA enc)":    {"method": "rrf",         "lambda": 0.5, "enc_key": "minilm_qa"},
}

print("=== RRF / Encoder comparison (first 100 docs for speed) ===")
sample_docs = D[:100]
Rset_sample = {d["doc_id"]: d["reference_summary"] for d in sample_docs}

for label, cfg in configs.items():
    S_cfg = {}
    for doc in sample_docs:
        S_cfg[doc["doc_id"]] = summarization(doc, 3, I, D, cfg)
    res = evaluation(S_cfg, Rset_sample, sample_docs, I,
                     {"mode": "extractive", "p": 3, "beta": 1.0})
    print(f"  {label:<22} Fβ={res['f_beta_mean']:.4f}  AUPRC={res['auprc_mean']:.4f}")